# 🧢 Detector de Acessórios de Cabeça em Fotos de Candidatos (Google Colab)

Este notebook executa a detecção de **acessórios de cabeça** (chapéus, bonés, toucas, tiaras, capacetes, turbantes, etc.) utilizando **YOLOS-Fashionpedia (Hugging Face Transformers)** com limiar de **confiança estrito superior a 90% (>0.90)**.

---

## 🛠️ Passo 1: Instalação das Bibliotecas Necessárias no Google Colab

In [ ]:
# Instalar Hugging Face Transformers, PyTorch, Pillow e Pandas
!pip install -q transformers pillow pandas torch torchvision

## 📁 Passo 2: Extrair Fotos de Amostra do GitHub (ou Montar Drive / Upload)
Escolha uma das opções abaixo no Colab. Por padrão, a **Opção A** baixa e extrai as fotos do repositório GitHub:
https://github.com/koiti/fotos_candidatos/tree/main/amostras

In [ ]:
# OPÇÃO A: Extrair fotos de amostra diretamente do GitHub (Recomendado)
# Repositório: https://github.com/koiti/fotos_candidatos/tree/main/amostras
import os
import shutil

!git clone https://github.com/koiti/fotos_candidatos.git

if os.path.exists('fotos_candidatos/amostras'):
    if os.path.exists('amostras'):
        shutil.rmtree('amostras')
    shutil.copytree('fotos_candidatos/amostras', 'amostras')
    print("-> Fotos de amostra extraídas com sucesso do GitHub para a pasta 'amostras'!")

In [ ]:
# OPÇÃO B: Montar o seu Google Drive (recomendado para pastas grandes como foto_cand2024_SP)
from google.colab import drive
drive.mount('/content/drive')

# Exemplo de caminho no seu Drive:
# input_path = '/content/drive/MyDrive/UFABC/2026-TOPICOS_DE_IA/foto_cand2024_SP'

In [ ]:
# OPÇÃO C: Fazer upload direto de um arquivo ZIP contendo as fotos (ex: amostras.zip)
from google.colab import files
import zipfile
import os

print('Faça o upload do seu arquivo .zip com as fotos:')
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('amostras')
        print('-> Arquivos extraídos com sucesso na pasta amostras!')

## 🧠 Passo 3: Código do Detector de Acessórios de Cabeça (YOLOS-Fashionpedia)

In [ ]:
import os
import json
import time
import torch
from pathlib import Path
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont, ImageOps
from transformers import pipeline
from IPython.display import display, Image as IPImage
import pandas as pd

# Verificar aceleração por GPU no Colab
device = 0 if torch.cuda.is_available() else -1
print(f'Dispositivo de inferência no Colab: {"GPU (CUDA)" if device == 0 else "CPU"}')

CLASSES_CABECA_MAP = {
    'hat': 'chapéu/boné',
    'headband, head covering, hair accessory': 'acessório de cabeça',
    'hood': 'capuz'
}

def executar_deteccao_acessorios_cabeca_colab(
    input_dir='amostras',
    output_dir='irregular_acessorios_cabeca',
    conf_thresh=0.90,
    batch_size=32
):
    target_input = Path(input_dir)
    target_output = Path(output_dir)
    target_output.mkdir(exist_ok=True, parents=True)

    fotos = sorted([
        f for f in target_input.iterdir()
        if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}
    ])

    if not fotos:
        print(f"Nenhuma foto encontrada na pasta '{target_input}'!")
        return None

    print(f"Analisando {len(fotos)} fotos em '{target_input}' com limiar de {conf_thresh:.0%}...")
    print('Carregando modelo YOLOS-Fashionpedia (Hugging Face)...')
    detector = pipeline('object-detection', model='valentinafevu/yolos-fashionpedia', device=device)

    total_irregulares = 0
    resultados_detalhados = []
    t_start = time.time()

    for b_idx in range(0, len(fotos), batch_size):
        bfiles = fotos[b_idx:b_idx + batch_size]
        bimgs = []
        vfiles = []
        for f in bfiles:
            try:
                img = Image.open(f)
                img = ImageOps.exif_transpose(img).convert('RGB')
                bimgs.append(img)
                vfiles.append(f)
            except Exception:
                pass

        if not bimgs:
            continue

        results = detector(bimgs, threshold=conf_thresh)

        for foto_path, img_rgb, res_list in zip(vfiles, bimgs, results):
            irregularidades = []
            for item in res_list:
                lbl_en = item['label']
                c_val = float(item['score'])
                box = item['box']
                if lbl_en in CLASSES_CABECA_MAP and c_val >= conf_thresh:
                    lbl_pt = CLASSES_CABECA_MAP[lbl_en]
                    irregularidades.append({
                        'classe_en': lbl_en,
                        'classe_pt': lbl_pt,
                        'confianca': round(c_val, 4),
                        'bbox': [box['xmin'], box['ymin'], box['xmax'], box['ymax']]
                    })

            if irregularidades:
                total_irregulares += 1
                draw = ImageDraw.Draw(img_rgb)
                for det in irregularidades:
                    b = det['bbox']
                    label_pt = det['classe_pt']
                    conf = det['confianca']
                    draw.rectangle(b, outline='red', width=4)
                    draw.text((b[0] + 5, max(0, b[1] - 15)), f'{label_pt} ({conf:.1%})', fill='red')
                img_rgb.save(target_output / foto_path.name)
                print(f" -> DETECTADO: {foto_path.name} | {irregularidades[0]['classe_pt']} ({irregularidades[0]['confianca']:.1%})")

            resultados_detalhados.append({
                'arquivo': foto_path.name,
                'status': 'irregular' if irregularidades else 'regular',
                'irregularidades': irregularidades
            })

    t_total = time.time() - t_start
    relatorio_data = {
        'data_analise': datetime.now().isoformat(),
        'modelo': 'valentinafevu/yolos-fashionpedia',
        'limiar_confianca': conf_thresh,
        'total_fotos': len(fotos),
        'total_irregulares': total_irregulares,
        'tempo_execucao_segundos': round(t_total, 2),
        'resultados': resultados_detalhados
    }

    relatorio_path = target_output / 'relatorio.json'
    with open(relatorio_path, 'w', encoding='utf-8') as f:
        json.dump(relatorio_data, f, ensure_ascii=False, indent=2)

    print('\n' + '=' * 70)
    print('VARREDURA CONCLUÍDA NO COLAB!')
    print(f'Total de fotos analisadas: {len(fotos)}')
    print(f'Irregulares salvas em "{target_output}": {total_irregulares}')
    print(f'Relatório salvo em: {relatorio_path}')
    return relatorio_data

## 🚀 Passo 4: Executar a Detecção

In [ ]:
# Executar a detecção com limiar de 90% de certeza
resultado = executar_deteccao_acessorios_cabeca_colab(
    input_dir='amostras',
    output_dir='irregular_acessorios_cabeca',
    conf_thresh=0.90
)

# Exibir dataframe com resultados irregulares
if resultado:
    df = pd.DataFrame(resultado['resultados'])
    display(df[df['status'] == 'irregular'])

## 🖼️ Passo 5: Visualizar Imagens Irregulares no Colab

In [ ]:
# Exibir fotos anotadas com bounding box vermelha
target = Path('irregular_acessorios_cabeca')
fotos_irregulares = sorted([f for f in target.glob('*.[jJ][pP]*[gG]')])

if fotos_irregulares:
    print(f'Exibindo {len(fotos_irregulares)} fotos irregulares:')
    for f in fotos_irregulares:
        print(f'📷 {f.name}')
        display(IPImage(filename=str(f), width=350))
else:
    print('Nenhuma imagem irregular encontrada com mais de 90% de certeza!')

## 💾 Passo 6: Baixar Resultados (.zip)

In [ ]:
# Compactar a pasta de resultados e fazer download para seu computador
!zip -r irregular_acessorios_cabeca.zip irregular_acessorios_cabeca
from google.colab import files
files.download('irregular_acessorios_cabeca.zip')